In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
if os.path.exists('/content/amend7_bundle.tgz'):
    D = '/content'
else:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && tar xzf {D}/amend7_bundle.tgz -C /content/w && cp {D}/inputs.json.gz /content/w/
!pip -q install accelerate
%cd /content/w
!python -m ddiood.amend7_jobs merge --inputs inputs.json.gz --delta amend7/delta_amend1.json.gz

In [ ]:
JOBS = [
 "a7-e7a-h5contam-s42",
 "a7-e7b-fixedownlabel-s42",
 "a7-e7c-names-s42",
 "a7-e7a-h5contam-s1",
 "a7-e7b-fixedownlabel-s1",
 "a7-e7c-names-s1",
 "a7-e7a-h5contam-s2",
 "a7-e7b-fixedownlabel-s2",
 "a7-e7c-names-s2",
 "a7-e7a-h5contam-s3",
 "a7-e7b-fixedownlabel-s3",
 "a7-e7c-names-s3",
 "a7-e7a-h5contam-s4",
 "a7-e7b-fixedownlabel-s4",
 "a7-e7c-names-s4"
]

In [ ]:
%cd /content/w
!bash -c "set -o pipefail; python -m ddiood.amend7_jobs run --workdir /content/w --out {D}/runs --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/E7_positive_control_part1.log | (grep --line-buffered -E '^(start|done|skip|FAILED)|Error|Traceback'; true)"

In [ ]:
import os, glob
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
failed = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.failed.txt'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])
print('failed:', failed)
for j in failed:
    print(j, open(f'{D}/runs/preds/{j}.failed.txt').read()[-1500:])
assert not failed, failed